# Phase 1, replicated: three conditions x three anchor draws

Every phase-1 number in this arm is currently a single training run at a
single anchor draw. 32/32 against 7/32 is too large to be noise, but
4/32, 5/32 and 7/32 across the three `anchors_v22` adapters are well
inside what seed variance could produce, and they have been read as if
they meant something. This puts error bars on all of it.

It also folds in the confound test, so that question gets answered with
replicates rather than once.

| condition | anchor worlds | targets on the optimal route | so far |
| --- | --- | --- | --- |
| **A** | bespoke `anchors.py` | 4/20 (20%) | 32/32, n=1 |
| **B** | `anchors_v22`, `silent_break` only | 20/20 (100%) | 7/32, n=1 |
| **C** | `anchors_v22`, four `irrelevant` per `silent_break` | 4/20 (20%) | never run |

C is the manipulation. Under v2.2 `irrelevant` sets its target link to
p = 0 exactly as `silent_break` does, so the target is dead in period B
in all 20 changed worlds under both B and C. The only thing that moves is
where the target sits relative to the optimal route.

Each condition runs at three anchor draws, `first_seed` 1000, 2000 and
3000, with the trainer seed varied alongside. Everything else is held at
the `augustopt` settings: 40 worlds, 140 examples, deterministic,
`PRES_REPEAT = 1`, `GRAD_ACCUM = 4`, 105 steps.

## What it settles

* whether 32/32 depends on the particular 40 worlds drawn at seed 1000
* whether the spread among the `anchors_v22` adapters is signal or noise
* whether route position is the confound, with C read against A and B
  rather than against a single number

In [ ]:
import glob
print(sorted(glob.glob("/kaggle/input/*")))
print([p for p in glob.glob("/kaggle/input/**/*anchors*.py", recursive=True)])

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
%pip install -q -U transformers peft bitsandbytes accelerate

## Preflight

In [ ]:
import sys, glob, json, time, gc, importlib.util, collections
import torch

def find_dir(marker, root="/kaggle/input"):
    hits = sorted(glob.glob(os.path.join(root, "**", marker), recursive=True),
                  key=lambda p: (p.count(os.sep), len(p)))
    if not hits:
        raise SystemExit(f"no {marker} under {root}")
    return os.path.dirname(hits[0])

RUN_TAG   = "replicate"
REPO_PATH = find_dir("resource_mdp.py")
EVAL_PATH = find_dir("anchors_v22.py")
GEN_PATH  = find_dir("gen_payloads.py")
OLD_HITS  = [p for p in glob.glob("/kaggle/input/**/anchors.py", recursive=True)
             if os.path.basename(p) == "anchors.py"]
if not OLD_HITS:
    raise SystemExit(
        "the original anchors.py is not attached. Condition A needs it. "
        "Add it to the ecpm eval dataset.")
OLD_ANCHORS = OLD_HITS[0]
OUT_DIR = "/kaggle/working"
print("repo:        ", REPO_PATH)
print("eval:        ", EVAL_PATH)
print("gen_payloads:", GEN_PATH)
print("old anchors: ", OLD_ANCHORS)

assert torch.cuda.is_available(), "no GPU: set Accelerator in the sidebar"
assert torch.cuda.device_count() == 1, (
    "CUDA_VISIBLE_DEVICES did not take; restart the kernel and run from the "
    "first cell")
CAP = torch.cuda.get_device_capability(0)
USE_BF16 = CAP[0] >= 8          # T4 is 7.5 and only emulates bf16
DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print(f"gpu: {torch.cuda.get_device_name(0)} | dtype: {DTYPE}")

In [ ]:
MODEL_NAME  = "Qwen/Qwen2.5-1.5B-Instruct"
DRAWS       = [1000, 2000, 3000]      # anchor first_seed per replicate
N_WORLDS    = 40
K           = 5
TOTAL_STEPS = 105
LR          = 1e-4
GRAD_ACCUM  = 4
MAX_LEN     = 2048
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.0
TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj",
           "gate_proj", "up_proj", "down_proj"]

CONDITIONS = {
    "A_bespoke":   None,                                  # anchors.py
    "B_onroute":   ("silent_break",),                     # 100% on-route
    "C_offroute":  ("irrelevant",) * 4 + ("silent_break",),  # 20% on-route
}

sys.path.insert(0, EVAL_PATH)
sys.path.insert(0, GEN_PATH)
import anchors_v22, gen_payloads as GP, ecpm_eval as E
rp = anchors_v22.load_env(REPO_PATH)
E.attach(REPO_PATH)
_spec = importlib.util.spec_from_file_location("old_anchors", OLD_ANCHORS)
oa = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(oa)
print("conditions:", list(CONDITIONS), "| draws:", DRAWS)
print(f"{len(CONDITIONS) * len(DRAWS)} runs at ~10 min each")

## Anchor builders

Condition A uses the original generator, which takes no condition
argument and is always deterministic. B and C use `anchors_v22` with
different condition cycles. All three produce 140 examples so the
optimizer sees the same amount of data.

In [ ]:
def build_examples(cond, draw):
    """(examples, stats) for one condition at one anchor draw."""
    if cond == "A_bespoke":
        worlds = oa.build_anchors(n_worlds=N_WORLDS, k=K, first_seed=draw)
        examples = [{"prompt": w["evidence"] + "\n\n" + q, "gold": a}
                    for w in worlds for _, q, a in w["items"]]
        on = off = 0
        for w in worlds:
            if not w["changed"]:
                continue
            adj = oa.make_world(w["seed"])
            route = {(s["node"], s["action"])
                     for s in oa.shortest(adj, oa.START, oa.GOAL, None)}
            if tuple(w["broken"]) in route:
                on += 1
            else:
                off += 1
        stats = {"on_route": on, "off_route": off, "conditions": "bespoke"}
    else:
        anchors_v22.CHANGED_CONDITIONS = CONDITIONS[cond]
        worlds = anchors_v22.build_anchor_set(
            rp, n_worlds=N_WORLDS, k=K, stochastic_share=0.0,
            first_seed=draw, preservation_changed_repeat=1)
        examples = anchors_v22.to_examples(worlds)
        on = off = dead = 0
        for w in worlds:
            if not w["changed"]:
                continue
            rec = rp.build_record(
                anchors_v22._scenario(w["seed"], w["condition"], K), True)
            if rec["change"].get("on_optimal_route"):
                on += 1
            else:
                off += 1
            dead += rec["change"].get("new_p") == 0.0
        assert dead == on + off, "not every target is dead in period B"
        stats = {"on_route": on, "off_route": off,
                 "conditions": anchors_v22.summarise(worlds)["conditions"]}
    assert len(examples) == 140, f"{cond}/{draw}: {len(examples)} examples"
    assert min(w["seed"] for w in worlds) > 79, "anchor seeds must clear 0-79"
    stats["examples"] = len(examples)
    return examples, stats

for cond in CONDITIONS:
    ex, st = build_examples(cond, DRAWS[0])
    rate = st["on_route"] / (st["on_route"] + st["off_route"])
    print(f"  {cond:12} examples {st['examples']}  on-route "
          f"{st['on_route']}/{st['on_route']+st['off_route']} ({rate:.0%})  "
          f"{st['conditions']}")

## Graded instances

Built once and reused by every run, so the only thing that differs
between runs is the adapter.

In [ ]:
rp_gp, _ = GP.load_env(REPO_PATH)
payloads = []
for seed in range(80):
    if len(payloads) >= 32:
        break
    try:
        payloads.append(GP.build_for_seed(rp_gp, seed, True, K, "silent_break"))
    except (ValueError, RuntimeError):
        pass
print(f"{len(payloads)} graded instances: {[p['seed'] for p in payloads]}")

## Train and evaluate

The base model is reloaded for each run rather than unloaded in place.
That costs about half a minute and removes any chance of one run's LoRA
state leaking into the next, which is the failure mode that would quietly
invalidate the whole comparison.

In [ ]:
from transformers import (AutoModelForCausalLM, AutoTokenizer,
                          BitsAndBytesConfig, Trainer, TrainingArguments)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from torch.utils.data import Dataset

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def as_ids(x):
    # apply_chat_template may return a BatchEncoding, which subclasses
    # UserDict rather than dict, so list(x) would give the keys
    if hasattr(x, "input_ids"):
        x = x.input_ids
    elif hasattr(x, "keys") and "input_ids" in x.keys():
        x = x["input_ids"]
    x = list(x)
    if x and isinstance(x[0], (list, tuple)):
        x = list(x[0])
    if not all(isinstance(t, int) for t in x):
        raise TypeError(f"expected token ids, got {type(x[0]).__name__}")
    return x

def encode(ex):
    pre = as_ids(tok.apply_chat_template(
        [{"role": "system", "content": E.SYSTEM},
         {"role": "user", "content": ex["prompt"]}],
        add_generation_prompt=True, tokenize=True))
    ans = as_ids(tok(ex["gold"] + tok.eos_token,
                     add_special_tokens=False)["input_ids"])
    return {"input_ids": pre + ans, "labels": [-100] * len(pre) + ans}

class Rows(Dataset):
    def __init__(self, rows):
        self.rows = rows
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        return dict(self.rows[i])

def collate(batch):
    n = max(len(b["input_ids"]) for b in batch)
    pad = tok.pad_token_id
    out = {"input_ids": [], "labels": [], "attention_mask": []}
    for b in batch:
        gap = n - len(b["input_ids"])
        out["input_ids"].append(b["input_ids"] + [pad] * gap)
        out["labels"].append(b["labels"] + [-100] * gap)
        out["attention_mask"].append([1] * len(b["input_ids"]) + [0] * gap)
    return {k: torch.tensor(v) for k, v in out.items()}

BNB = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=DTYPE,
                         bnb_4bit_use_double_quant=True)

def train_and_eval(cond, draw, seed_idx):
    examples, stats = build_examples(cond, draw)
    enc = [encode(e) for e in examples]
    assert max(len(x["input_ids"]) for x in enc) <= MAX_LEN

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=BNB, device_map={"": 0})
    model = prepare_model_for_kbit_training(model,
                                            use_gradient_checkpointing=True)
    model.config.use_cache = False
    model = get_peft_model(model, LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
        bias="none", task_type="CAUSAL_LM", target_modules=TARGETS))
    res = Trainer(
        model=model,
        args=TrainingArguments(
            output_dir=f"{OUT_DIR}/rep_tmp",
            per_device_train_batch_size=1,
            gradient_accumulation_steps=GRAD_ACCUM,
            max_steps=TOTAL_STEPS, learning_rate=LR,
            warmup_steps=max(1, TOTAL_STEPS // 20),
            lr_scheduler_type="cosine", logging_strategy="no",
            save_strategy="no", report_to=[], seed=42 + seed_idx,
            bf16=USE_BF16, fp16=not USE_BF16,
            gradient_checkpointing=True,
            gradient_checkpointing_kwargs={"use_reentrant": False}),
        train_dataset=Rows(enc), data_collator=collate).train()

    model.eval()
    model.gradient_checkpointing_disable()
    model.config.use_cache = True

    def generate(messages, n):
        e = tok.apply_chat_template(messages, add_generation_prompt=True,
                                    return_tensors="pt", return_dict=True)
        e = {k: v.to(model.device) for k, v in e.items()}
        with torch.no_grad():
            o = model.generate(**e, max_new_tokens=n, do_sample=False,
                               pad_token_id=tok.eos_token_id)
        return tok.decode(o[0, e["input_ids"].shape[1]:],
                          skip_special_tokens=True)

    rows = E.run_arm(payloads, generate, arm="arm_c", mode="single",
                     probes=["localization"], verbose=False)
    ok = [r["seed"] for r in rows if r["scored"].get("correct")]
    node = sum(1 for r in rows if r["parsed"].get("node") == r["target"][0])

    del model
    gc.collect()
    torch.cuda.empty_cache()
    return {"condition": cond, "draw": draw, "n": len(rows),
            "localize": len(ok), "node_level": node,
            "correct_seeds": sorted(ok),
            "final_loss": res.metrics.get("train_loss"), **stats}, rows

In [ ]:
res_path = os.path.join(OUT_DIR, f"results_{RUN_TAG}.jsonl")
raw_path = os.path.join(OUT_DIR, f"raw_{RUN_TAG}.jsonl")
results, done = [], set()
if os.path.exists(res_path):
    for line in open(res_path):
        r = json.loads(line)
        results.append(r)
        done.add((r["condition"], r["draw"]))
    print(f"resuming, {len(results)} runs already saved")

t_all = time.time()
for si, draw in enumerate(DRAWS):
    for cond in CONDITIONS:
        if (cond, draw) in done:
            print(f"{cond} draw {draw}: done")
            continue
        t0 = time.time()
        summary, rows = train_and_eval(cond, draw, si)
        with open(res_path, "a") as f:
            f.write(json.dumps(summary) + "\n")
        with open(raw_path, "a") as f:
            for r in rows:
                r["condition"], r["draw"] = cond, draw
                f.write(json.dumps(r) + "\n")
        results.append(summary)
        print(f"{cond:12} draw {draw}: localize {summary['localize']}/"
              f"{summary['n']}  node {summary['node_level']}/{summary['n']}  "
              f"loss {summary['final_loss']:.4f}  "
              f"({(time.time()-t0)/60:.1f} min)")
print(f"\ntotal {(time.time()-t_all)/60:.1f} min")

## Results

In [ ]:
import pandas as pd

df = pd.DataFrame(results).sort_values(["condition", "draw"])
display(df[["condition", "draw", "on_route", "localize", "node_level",
            "final_loss"]])

print(f"\n{'condition':14} {'on-route':>9} {'localize by draw':>22} "
      f"{'mean':>7} {'range':>9}")
for cond in CONDITIONS:
    sub = [r for r in results if r["condition"] == cond]
    if not sub:
        continue
    vals = [r["localize"] for r in sub]
    rate = sub[0]["on_route"] / (sub[0]["on_route"] + sub[0]["off_route"])
    print(f"  {cond:12} {rate:>8.0%} "
          f"{str(vals):>22} {sum(vals)/len(vals):>7.1f} "
          f"{min(vals)}-{max(vals):<7}")

print("\nreference points on the same 32 instances:")
print("  evidence-only rule          32/32")
print("  untrained, evidence in prompt  4/32")
print("\nread C against A and B. If C sits with A, route position is the")
print("confound and anchors_v22 is fixable by mixing conditions. If C sits")
print("with B, it is not, and the mechanism paragraph comes out of 6.2.")

json.dump(results, open(os.path.join(OUT_DIR, f"summary_{RUN_TAG}.json"), "w"),
          indent=1)

In [ ]:
# do the conditions separate cleanly, or do the ranges overlap?
import itertools
pairs = list(itertools.combinations(CONDITIONS, 2))
print("range overlap between conditions:")
for a, b in pairs:
    va = [r["localize"] for r in results if r["condition"] == a]
    vb = [r["localize"] for r in results if r["condition"] == b]
    if not va or not vb:
        continue
    overlap = not (max(va) < min(vb) or max(vb) < min(va))
    print(f"  {a:12} {min(va)}-{max(va):<3} vs {b:12} {min(vb)}-{max(vb):<3}"
          f"   {'OVERLAP' if overlap else 'separated'}")
print("\nnon-overlapping ranges at n=3 are weak evidence but enough to say")
print("the difference is not a draw artefact. Overlap means the single-run")
print("numbers reported so far cannot be distinguished.")

## Package

In [ ]:
import zipfile, shutil
shutil.rmtree(f"{OUT_DIR}/rep_tmp", ignore_errors=True)
ZIP_PATH = f"{OUT_DIR}/{RUN_TAG}_all.zip"
n = 0
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    for path in sorted(glob.glob(f"{OUT_DIR}/**/*", recursive=True)):
        if (not os.path.isfile(path) or os.path.basename(ZIP_PATH) in path
                or ".ipynb_checkpoints" in path):
            continue
        z.write(path, os.path.relpath(path, OUT_DIR))
        n += 1
print(f"{n} files -> {ZIP_PATH} ({os.path.getsize(ZIP_PATH)/1e6:.1f} MB)")